In [25]:
import pandas as pd
import re
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import numbers

In [14]:
main_file = Path(r"D:\Proyek FEB\Publikasi internasional.xlsx")
scrap_file = Path(r"D:\Proyek FEB\lengkapin data\sinta_q_citation_results.xlsx")

df_main = pd.read_excel(main_file)
df_scrap = pd.read_excel(scrap_file)

COL_TITLE = "Title"
COL_Q = "SCOPUS Q"
COL_CITATION = "SCOPUS SITASI"
COL_YEAR = "Tahun"

In [15]:
def is_empty_value(value):
    """
    Data utama memakai tanda ? sebagai nilai kosong.
    Jadi ?, NaN, kosong, -, n/a dianggap missing.
    """
    if pd.isna(value):
        return True
    
    text = str(value).strip()
    
    if text == "":
        return True
    
    if text.lower() in [
        "nan",
        "none",
        "null",
        "-",
        "?",
        "??",
        "???",
        "n/a",
        "na",
        "tidak ada",
        "not found"
    ]:
        return True
    
    return False


def normalize_title(value):
    """
    Normalisasi judul untuk key mapping.
    """
    if pd.isna(value):
        return ""
    
    text = str(value).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", "", text)
    
    return text.strip()


def clean_q(value):
    """
    Standarisasi nilai Q.
    """
    if is_empty_value(value):
        return None
    
    text = str(value).strip()
    
    if re.search(r"\bno-?Q\b", text, flags=re.IGNORECASE):
        return "no-Q"
    
    match = re.search(r"\bQ\s*([1-4])\b", text, flags=re.IGNORECASE)
    
    if match:
        return f"Q{match.group(1)}"
    
    return text


def clean_citation(value):
    """
    Standarisasi sitasi menjadi integer.
    """
    if is_empty_value(value):
        return None
    
    text = str(value).strip()
    
    match = re.search(r"(\d+)", text)
    
    if match:
        return int(match.group(1))
    
    return None

In [16]:
df_scrap_clean = df_scrap.copy()

# Pakai hanya hasil yang ditemukan di SINTA
df_scrap_clean = df_scrap_clean[df_scrap_clean["status"] == "found"].copy()

# Key untuk mapping
df_scrap_clean["title_key"] = df_scrap_clean["input_title"].apply(normalize_title)

# Bersihkan Q dan sitasi
df_scrap_clean["scopus_q_clean"] = df_scrap_clean["scopus_q"].apply(clean_q)
df_scrap_clean["scopus_citation_clean"] = df_scrap_clean["scopus_citation"].apply(clean_citation)

# Hapus duplikat title_key kalau ada
df_scrap_clean = df_scrap_clean.drop_duplicates(subset=["title_key"], keep="first")

df_scrap_clean[
    [
        "input_title",
        "matched_title",
        "scopus_q",
        "scopus_q_clean",
        "scopus_citation",
        "scopus_citation_clean",
        "status"
    ]
].head()

,input_title,matched_title,scopus_q,scopus_q_clean,scopus_citation,scopus_citation_clean,status
0,Ethnic identity and internal migration decisio...,Ethnic identity and internal migration decisio...,Q1,Q1,24.0,24,found
1,The role of service quality within Indonesian ...,The role of service quality within Indonesian ...,Q2,Q2,60.0,60,found
2,Developing Islamic crowdfunding website platfo...,Developing Islamic crowdfunding website platfo...,Q2,Q2,48.0,48,found
3,Exploration of pilgrimage tourism in Indonesia,Exploration of pilgrimage tourism in Indonesia,Q2,Q2,33.0,33,found
4,A critical assessment of retail sovereign suku...,A critical assessment of retail sovereign suku...,Q2,Q2,9.0,9,found


In [17]:
df_merged = df_main.copy()

# Simpan salinan Tahun asli supaya bisa dipastikan tidak berubah
tahun_before = df_merged[COL_YEAR].copy()

# Key mapping
df_merged["title_key"] = df_merged[COL_TITLE].apply(normalize_title)

In [18]:
before_missing = df_merged[[COL_Q, COL_CITATION]].applymap(is_empty_value).sum()

before_missing

C:\Users\user\AppData\Local\Temp\ipykernel_5172\927578189.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  before_missing = df_merged[[COL_Q, COL_CITATION]].applymap(is_empty_value).sum()


SCOPUS Q         2153
SCOPUS SITASI    2153
dtype: int64

In [19]:
scrap_map = (
    df_scrap_clean
    .set_index("title_key")
    .to_dict(orient="index")
)

In [20]:
updated_q_count = 0
updated_citation_count = 0

for idx, row in df_merged.iterrows():
    key = row["title_key"]
    
    if key not in scrap_map:
        continue
    
    scrap_row = scrap_map[key]
    
    # Isi SCOPUS Q hanya jika kosong / ?
    if is_empty_value(row[COL_Q]):
        q_value = scrap_row.get("scopus_q_clean")
        
        if not is_empty_value(q_value):
            df_merged.at[idx, COL_Q] = q_value
            updated_q_count += 1
    
    # Isi SCOPUS SITASI hanya jika kosong / ?
    if is_empty_value(row[COL_CITATION]):
        citation_value = scrap_row.get("scopus_citation_clean")
        
        if not is_empty_value(citation_value):
            df_merged.at[idx, COL_CITATION] = int(citation_value)
            updated_citation_count += 1

print("SCOPUS Q terisi:", updated_q_count)
print("SCOPUS SITASI terisi:", updated_citation_count)

SCOPUS Q terisi: 629
SCOPUS SITASI terisi: 629


In [21]:
df_merged[COL_YEAR] = tahun_before

In [22]:
df_merged = df_merged.drop(columns=["title_key"])

In [23]:
after_missing = df_merged[[COL_Q, COL_CITATION]].applymap(is_empty_value).sum()

comparison = pd.DataFrame({
    "before_missing": before_missing,
    "after_missing": after_missing,
    "filled": before_missing - after_missing
})

comparison

C:\Users\user\AppData\Local\Temp\ipykernel_5172\3620706693.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  after_missing = df_merged[[COL_Q, COL_CITATION]].applymap(is_empty_value).sum()


,before_missing,after_missing,filled
SCOPUS Q,2153,1524,629
SCOPUS SITASI,2153,1524,629


In [27]:
output_file = "checkpoint_1_merged_sitasi&Q_sinta.xlsx"

# Export dulu
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_merged.to_excel(writer, index=False, sheet_name="Data")

# Buka lagi workbook untuk atur format kolom Tahun
wb = load_workbook(output_file)
ws = wb["Data"]

# Cari posisi kolom Tahun
header_row = 1
tahun_col_idx = None

for cell in ws[header_row]:
    if cell.value == "Tahun":
        tahun_col_idx = cell.column
        break

if tahun_col_idx is None:
    raise ValueError("Kolom Tahun tidak ditemukan")

# Paksa format tanggal tanpa jam
for row in range(2, ws.max_row + 1):
    cell = ws.cell(row=row, column=tahun_col_idx)
    
    if cell.value is not None:
        cell.number_format = "m/d/yyyy"   # contoh: 1/11/2019

wb.save(output_file)

print(f"Saved: {output_file}")

Saved: checkpoint_1_merged_sitasi&Q_sinta.xlsx
